<a href="https://colab.research.google.com/github/jacobzeitlin/anki-addons/blob/main/Program_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Dependencies

try:
    import playwright
except:
    !pip install --quiet playwright --progress-bar off
    !playwright install --with-deps

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

import asyncio
import pandas as pd
import re


Installing dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [962 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Ign:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Get:13 https://ppa.launchpadcontent.net/g

In [ ]:
# @title ORIN

url = "https://orin.aoassn.org/#/search"

async def extract_orin():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless = True)
        page = await browser.new_page()

        await page.goto(url)
        await page.wait_for_selector("button.btn-primary")
        await page.click("button.btn-primary")
        await page.wait_for_selector("div.p-3.border-bottom")

        html = await page.content()

        await browser.close()

        soup = BeautifulSoup(html, "html.parser")

        programs = soup.find_all("div", class_= "p-3 border-bottom")
        data = []
        names = []
        locations = []

        for program in programs:
            name = program.find("span", class_ = "h5 text-primary").text.strip()
            location = program.find("p", class_ = "m-0 mb-1 text-dark").text.strip()
            details = [detail.text.strip() for detail in program.find_all("div", class_ = "col")]

            names.append(name)
            locations.append(location)
            data.append(details)

        data = pd.DataFrame(data)
        names = pd.Series(names)
        locations = pd.Series(locations)

        return [names, locations, data]

all = await extract_orin()

d = all[2]

program_summaries = pd.DataFrame(index = range(len(d)), columns = pd.concat([d.iloc[0, 1:6], d.iloc[0, 11:17]]))

program_summaries.iloc[:, :5] = d.iloc[:, 6:11]
program_summaries.iloc[:, 5:11] = d.iloc[:, 17:23]

program_summaries.insert(0, "Program", all[0])
program_summaries.insert(1, "Updated", d.iloc[:, 0])
program_summaries.insert(2, "Location", all[1])

program_summaries["Updated"] = program_summaries.Updated.str.replace("Last Updated ", "")
program_summaries["Updated"] = pd.to_datetime(program_summaries.Updated, format = "%b %d, %Y, %I:%M %p")


In [4]:
# @title STAR

url = "https://app.powerbi.com/groups/me/reports/903aad6a-750d-455d-9f09-2b40a19e6757/ReportSection?ctid=9d418695-71ac-4c31-b5b2-c196c8ec3c8a"

async def extract_star():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless = True)
        context = await browser.new_context(device_scale_factor = 2)
        page = await context.new_page()

        await page.goto(url)
        await page.wait_for_selector("#email", timeout = 14000)
        await page.fill("#email", "jhz4001@med.cornell.edu")
        await page.click("#submitBtn")
        await page.wait_for_selector("input[name='passwd']", timeout = 14000)
        await page.fill("input[name='passwd']", "grunge lint pucker acorn")
        await page.click("input[type='submit']")

        print("\nWaiting for root content...")

        prob = await page.content()
        soup = BeautifulSoup(prob, "html.parser")

        for tag in soup.find_all(True):
                del tag["style"]

        print(soup)
        return soup

        await page.wait_for_selector("#rootContent", timeout = 28000)

        print("\nWaiting for specialty...")

        await page.click("div.slicer-dropdown-menu")
        await page.click("text=Orthopaedic Surgery")

        print("\nWaiting for query to be displayed...")

        try:
            await page.wait_for_selector("button[data-testid='focus-mode-btn']", timeout = 28000)
            print("\nWaiting for focus mode...")
            await page.click("button[data-testid='focus-mode-btn']", timeout = 14000)
        except Exception as e:
            print("\nCould not focus:\n" + str(e))
            # return await page.content()

        print("\nStarting to scroll and collect data...")

        apps = []  # List to hold all collected data

        while True:
            # Get the current HTML content
            html = await page.content()
            soup = BeautifulSoup(html, "html.parser")

            # Remove inline styles for consistency
            for tag in soup.find_all(True):
                del tag["style"]

            # Find and store the data you need (example: finding a specific div)
            # visual_container = soup.find("div", class_ = "visualContainer unselectable readMode hideBorder droppableElement ui-droppable selected swipeable-blocked popOut noVisualTitle")
            visual_container = soup.find_all("div", class_ = re.compile(r"\bvisualContainer\b.*\bui-droppable\b"))

            if visual_container:
                apps.append(visual_container)
            else:
                print("\nNo visual container found on this page.")
                print(len(apps))
                print("\nClosing browser...")
                await browser.close()
                return html

            # Scroll down the page
            previous_height = await page.evaluate("document.body.scrollHeight")
            await page.evaluate("window.scrollBy(0, window.innerHeight);")

            # Wait for the page to load new content
            await page.wait_for_timeout(1000)  # Adjust if needed

            # Check if we've reached the bottom of the page
            new_height = await page.evaluate("document.body.scrollHeight")
            if new_height == previous_height:
                print("\nReached the bottom of the page.")
                break  # Exit the loop if no new content is loaded

        print("\nScrolling complete, closing browser...")

        await browser.close()

        print("\nData collection complete.")

        return apps

        # print("\nWaiting for focused content...")

        # html = await page.content()

        # print("\nClosing browser...")

        # await browser.close()

        # print("\nStarting soup...")

        # soup = BeautifulSoup(html, "html.parser")

        # for tag in soup.find_all(True):
        #     del tag["style"]

        # apps = soup.find("div", class_ = "visualContainer unselectable readMode hideBorder droppableElement ui-droppable selected swipeable-blocked popOut noVisualTitle")
        # print("\nDone.")

        # return apps

html_returned = await extract_star()

# with open("html_returned.html", "w") as f:
#     f.write(str(html_returned))



Waiting for root content...
<!DOCTYPE html>
<html class="" dir="ltr" lang="en"><head>
<title>Sign in to your account</title>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1.0, maximum-scale=2.0, user-scalable=yes" name="viewport"/>
<meta content="no-cache" http-equiv="Pragma"/>
<meta content="-1" http-equiv="Expires"/>
<link crossorigin="" href="https://aadcdn.msauth.net" rel="preconnect"/>
<meta content="on" http-equiv="x-dns-prefetch-control"/>
<link href="//aadcdn.msauth.net" rel="dns-prefetch"/>
<link href="//aadcdn.msftauth.net" rel="dns-prefetch"/>
<meta content="ConvergedSignIn" name="PageID"/>
<meta content="" name="SiteID"/>
<meta content="1033" name="ReqLC"/>
<meta content="en-US" name="LocLC"/>
<meta content="telephone=no" name="format-detection"/>
<noscript>
<meta content="0; URL=https://login.microsoftonline.com/jsdisabled" http-equiv="Refresh">
<

In [ ]:
len(html_returned[0])


21

In [ ]:
print(html_returned[0][1])

#Simulate a user scrolling down within Chromium



<div _ngcontent-ng-c1927612466="" aria-hidden="false" aria-label="Applicants " aria-roledescription="" class="visualContainer unselectable readMode hideBorder droppableElement ui-droppable" extended-shortcut-scope="" ng-non-bindable="" role="group" skip-children-focus="" tab-order="7000" tabindex="0" toolbar-anchor="" touch-action="auto"><p _ngcontent-ng-c1927612466="" aria-hidden="true" class="visualsEnterHint ng-star-inserted" id="visualsEnterHint-4a5eeb0285e69deb3912" localize="VisualContainer_VisualsEnterHint_OnView" role="tooltip">Press Enter to explore data</p><!-- --><!-- --><p _ngcontent-ng-c1927612466="" aria-hidden="true" class="visualsEnterHint" id="visualsLabel-4a5eeb0285e69deb3912" role="tooltip">Applicants </p><div _ngcontent-ng-c1927612466="" class="visualContainerBorder themableBorderColor ng-star-inserted"></div><!-- --><div _ngcontent-ng-c1927612466="" class="visualContainerScreen ng-star-inserted"></div><!-- --><!-- --><!-- --><!-- --><!-- --><!-- --><div _ngcontent-

In [ ]:
url = "https://www.doximity.com/residency/programs?specialtyKey=bd234238-6960-4260-9475-1fa18f58f092-orthopaedic-surgery&sortByKey=reputation&trainingEnvironmentKey=&intendedFellowshipKey="

# async def extract_rows():
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless = True)
#         page = await browser.new_page()

#         await page.goto(url)


In [ ]:
# program_summaries.head()


In [ ]:
# program_summaries.to_csv("program_summaries.csv", index = False)
